# Uri Watch: fine-tune Granite TinyTimeMixer (TTM) on Texas county outages

**What this notebook does:** fine-tunes IBM's pretrained TTM to forecast each county's
outage percentage for the next 96 hours, using weather (known ahead) and statewide grid
demand (past values only) as extra inputs. It then compares the result against a naive
"nothing changes" baseline.

**Run order:** top to bottom. Leave `USE_SYNTHETIC = True` for the first pass. That is a
dry run on fake data that checks the library, the model download, and training speed on this
VM before the real table exists. Switch it to `False` when `county_hourly.csv` is ready.

**Data contract for the real table** (one row per county per hour, saved as CSV):

| column | meaning |
|---|---|
| `fips` | county FIPS code as a string, for example `"48113"` |
| `timestamp` | hourly, UTC, timezone-naive |
| `pct_out` | customers out / customers served x 100 (EAGLE-I) |
| `temp_c`, `wind_ms`, `gust_ms`, `precip_mm`, `snow_cm` | Open-Meteo weather for the county centroid |
| `grid_demand_mwh` | statewide ERCOT demand for that hour (same value for every county) |

Every county must have every hour in the range (no gaps).

(We use CSV rather than Parquet because this LinuxONE / s390x VM has no `pyarrow` wheel.)

**Caveat to repeat in the demo:** historical weather here is observed, so it stands in for a
forecast. Real responders would have forecasts, which are less accurate.

## 1. Environment check

In [ ]:
import sys, platform, importlib
from pathlib import Path

# This LinuxONE / s390x VM has no pyarrow wheel, so HuggingFace `datasets` cannot be
# installed. tsfm_public only needs its `Dataset` class for type hints / isinstance
# checks (our DataFrame path never uses it), so a tiny local stub stands in. Put the
# project `stubs/` dir on sys.path *before* importing tsfm_public, so the notebook
# works both in Jupyter and when run headless. (No effect if real `datasets` exists.)
_stub = Path.home() / "outage_project" / "stubs"
if _stub.is_dir() and str(_stub) not in sys.path:
    sys.path.insert(0, str(_stub))

print("python", sys.version.split()[0], "| arch", platform.machine())
REQUIRED = ["numpy", "pandas", "torch", "transformers", "tsfm_public"]
OPTIONAL = ["pyarrow"]   # no s390x wheel on this VM; the notebook uses CSV instead of Parquet
for name in REQUIRED:
    try:
        m = importlib.import_module(name)
        print(f"{name:12s} OK       {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"{name:12s} MISSING  ({type(e).__name__}: {e})")
for name in OPTIONAL:
    try:
        m = importlib.import_module(name)
        print(f"{name:12s} OK       {getattr(m, '__version__', '')} (optional)")
    except Exception:
        print(f"{name:12s} absent   (optional; not needed, we read/write CSV)")

If `tsfm_public` is missing, it comes from the `granite-tsfm` package. Your teammate's model runs
in this same environment, so **do not install blindly**. A new `torch` or `transformers` version
could break her setup. In a terminal, first run:

```
source ~/jupyter-env/bin/activate
pip install --dry-run granite-tsfm
```

The dry run lists what pip would change. If it wants to replace `torch`, `numpy` or
`transformers`, stop and coordinate with her before installing.

## 2. Configuration

In [ ]:
from pathlib import Path

PROJECT = Path.home() / "outage_project"
DATA_PATH = PROJECT / "data" / "county_hourly.csv"   # CSV, not Parquet (no pyarrow on s390x)
OUT_DIR = PROJECT / "models" / "ttm_outage_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_SYNTHETIC = True      # True = dry run on fake data. Set False for the real table.
COUNTY_SAMPLE = 60        # real data only: random subset of counties for the first run (None = all)

ID_COL, TIME_COL, TARGET = "fips", "timestamp", "pct_out"
CONTROL_COLS = ["temp_c", "wind_ms", "gust_ms", "precip_mm", "snow_cm"]   # known ahead (forecast stand-in)
CONDITIONAL_COLS = ["grid_demand_mwh"]                                   # past values only

CONTEXT_LEN, PRED_LEN = 512, 96       # 512 hours of history -> next 96 hours

# Chronological split. Everything before TRAIN_END trains, TRAIN_END..VALID_END picks the best
# checkpoint, VALID_END..TEST_END is the held-out Uri month (never used for any choice).
TRAIN_END, VALID_END, TEST_END = "2021-01-15", "2021-02-01", "2021-03-01"

EVENT_PCT = 2.0           # "serious outage" = at least this % of a county's customers out
FREEZE_BACKBONE = True    # True = few-shot style (fast). False = full fine-tune (slower).
MAX_STEPS = 1500          # hard cap on training steps, so run time is bounded
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
TEST_STRIDE = 1           # evaluate every Nth test window (raise to 4 or 8 if evaluation is slow)
SEED = 42

## 3. Load data (synthetic dry run or the real table)

In [ ]:
import numpy as np
import pandas as pd


def make_synthetic(n_counties=8, start="2018-10-01", end="2021-03-01", seed=0):
    # Fake counties with a freeze-driven outage signal and a huge Feb 2021 cold event.
    rng = np.random.default_rng(seed)
    ts = pd.date_range(start, end, freq="h", inclusive="left")
    n = len(ts)
    doy, hod = ts.dayofyear.to_numpy(), ts.hour.to_numpy()
    seasonal = 17 - 12 * np.cos(2 * np.pi * (doy - 15) / 365) + 4 * np.sin(2 * np.pi * (hod - 9) / 24)
    drop = np.zeros(n)
    for year in range(ts.year.min(), ts.year.max() + 1):
        for _ in range(3):
            s = pd.Timestamp(year=year, month=int(rng.choice([12, 1, 2])), day=int(rng.integers(1, 25)))
            a = int(np.searchsorted(ts.values, np.datetime64(s)))
            drop[a:a + int(rng.integers(36, 96))] += rng.uniform(6, 12)
    a = int(np.searchsorted(ts.values, np.datetime64("2021-02-10")))
    b = int(np.searchsorted(ts.values, np.datetime64("2021-02-18")))
    drop[a:b] += 24
    state_temp = seasonal - drop
    demand = 40000 + 700 * np.abs(state_temp - 18) + rng.normal(0, 500, n)
    frames = []
    for c in range(n_counties):
        temp = state_temp + rng.normal(0, 1.0, n) + rng.normal(0, 2)
        freeze = pd.Series((temp < 0).astype(float)).rolling(24, min_periods=1).sum().to_numpy()
        raw = rng.uniform(0.3, 1.5) * np.clip((freeze - 6) / 18, 0, None) ** 2 * 8.0
        lagged = np.concatenate([np.zeros(3), raw[:-3]])
        pct = np.clip(lagged + rng.normal(0, 0.05, n) * (lagged > 0), 0, 100)
        wind = np.abs(rng.normal(4, 2, n))
        precip = np.where(rng.random(n) < 0.05, rng.exponential(2, n), 0.0)
        frames.append(pd.DataFrame({
            ID_COL: f"48{2 * c + 1:03d}", TIME_COL: ts, TARGET: pct.round(4),
            "temp_c": temp, "wind_ms": wind, "gust_ms": wind * rng.uniform(1.3, 1.8, n),
            "precip_mm": precip, "snow_cm": np.where((temp < 0) & (precip > 0), precip * 0.8, 0.0),
            "grid_demand_mwh": demand,
        }))
    return pd.concat(frames, ignore_index=True)


def check_contract(df):
    need = [ID_COL, TIME_COL, TARGET] + CONTROL_COLS + CONDITIONAL_COLS
    missing = [c for c in need if c not in df.columns]
    assert not missing, f"missing columns: {missing}"
    df = df[need].copy()
    df[ID_COL] = df[ID_COL].astype(str)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    if df[TIME_COL].dt.tz is not None:
        df[TIME_COL] = df[TIME_COL].dt.tz_convert("UTC").dt.tz_localize(None)
    df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
    g = df.groupby(ID_COL)[TIME_COL]
    assert g.size().nunique() == 1, "counties have different numbers of rows; fill the hourly grid first"
    assert g.min().nunique() == 1 and g.max().nunique() == 1, "counties cover different time ranges"
    first = df[df[ID_COL] == df[ID_COL].iloc[0]][TIME_COL]
    assert (first.diff().dropna() == pd.Timedelta(hours=1)).all(), "timestamps are not strictly hourly"
    nan_frac = df[need].isna().mean()
    bad = nan_frac[nan_frac > 0.01]
    assert bad.empty, f"more than 1% missing values in: {bad.to_dict()}"
    if nan_frac.max() > 0:
        df[CONTROL_COLS + CONDITIONAL_COLS] = df.groupby(ID_COL)[CONTROL_COLS + CONDITIONAL_COLS].transform(
            lambda s: s.ffill().bfill())
        df[TARGET] = df[TARGET].fillna(0.0)
        print("filled small gaps:", nan_frac[nan_frac > 0].round(5).to_dict())
    return df


if USE_SYNTHETIC:
    raw_df = make_synthetic()
    print("SYNTHETIC DRY RUN. These numbers mean nothing about real outages.")
else:
    raw_df = pd.read_csv(DATA_PATH, dtype={ID_COL: str})   # CSV, not Parquet (no pyarrow on s390x)

df = check_contract(raw_df)
if (not USE_SYNTHETIC) and COUNTY_SAMPLE:
    keep = np.random.default_rng(SEED).choice(df[ID_COL].unique(), size=min(COUNTY_SAMPLE, df[ID_COL].nunique()),
                                              replace=False)
    df = df[df[ID_COL].isin(keep)].reset_index(drop=True)
print(f"{df[ID_COL].nunique()} counties, {df[TIME_COL].min()} -> {df[TIME_COL].max()}, {len(df):,} rows")
print(df[TARGET].describe().round(3).to_string())

## 4. Prepare the model table

The target is `log1p(pct_out)`, because most hours are zero and a few are huge. I scale every
column with **one set of statistics shared by all counties** (computed on training hours only), so
predictions stay comparable across counties when you rank them later.

In [ ]:
df = df[df[TIME_COL] < pd.Timestamp(TEST_END)].copy()
df["y"] = np.log1p(df[TARGET].clip(lower=0))
SCALE_COLS = ["y"] + CONTROL_COLS + CONDITIONAL_COLS

train_mask = df[TIME_COL] < pd.Timestamp(TRAIN_END)
stats = df.loc[train_mask, SCALE_COLS].agg(["mean", "std"]).T
stats["std"] = stats["std"].replace(0, 1.0).fillna(1.0)
for c in SCALE_COLS:
    df[c] = ((df[c] - stats.loc[c, "mean"]) / stats.loc[c, "std"]).astype("float32")
df = df[[ID_COL, TIME_COL, "y"] + CONTROL_COLS + CONDITIONAL_COLS]

Y_MEAN, Y_STD = float(stats.loc["y", "mean"]), float(stats.loc["y", "std"])
print(stats.round(4).to_string())
print("model table:", df.shape, "| memory MB:", round(df.memory_usage(deep=True).sum() / 1e6, 1))

## 5. Evaluation helpers (pure numpy/pandas, no model needed)

In [ ]:
def to_pct(a, mean, std):
    return np.clip(np.expm1(np.asarray(a) * std + mean), 0, 100)


def auc_score(scores, labels):
    labels = np.asarray(labels).astype(bool)
    n_pos, n_neg = int(labels.sum()), int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = pd.Series(np.asarray(scores)).rank(method="average").to_numpy()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def r_precision(scores, labels):
    # Flag exactly as many windows as there are real events; then precision equals recall.
    scores, labels = np.asarray(scores), np.asarray(labels).astype(bool)
    k = int(labels.sum())
    if k == 0:
        return float("nan")
    top = np.argsort(-scores, kind="stable")[:k]
    return float(labels[top].sum() / k)


def eval_report(pred_s, last_s, truth_s, mean, std, event_pct, horizons=(6, 24, 96)):
    # pred_s/truth_s: (N, horizon) scaled log-space. last_s: (N,) last observed value before the forecast.
    pred, truth, last = to_pct(pred_s, mean, std), to_pct(truth_s, mean, std), to_pct(last_s, mean, std)
    pers = np.repeat(last[:, None], pred.shape[1], axis=1)
    rows = []
    for h in horizons:
        h = min(h, pred.shape[1])
        event = truth[:, :h].max(axis=1) >= event_pct
        for name, p in (("TTM", pred), ("persistence", pers)):
            s = p[:, :h].max(axis=1)
            rows.append(dict(horizon_h=h, model=name, mae_pct=float(np.abs(p[:, :h] - truth[:, :h]).mean()),
                             auc=auc_score(s, event), hit_rate=r_precision(s, event),
                             event_rate=float(event.mean())))
    return pd.DataFrame(rows)

## 6. Build the TTM datasets

This section needs `granite-tsfm`. Different library versions treat the start of the validation and
test splits differently (whether they add the 512 hours of context themselves), so the cell checks the
window counts against what it expects and adjusts automatically. If it raises an error, copy the
message and send it to me.

In [ ]:
import inspect, torch
from tsfm_public import TimeSeriesPreprocessor, get_datasets


def filter_kwargs(fn, kwargs, required=()):
    params = inspect.signature(fn).parameters
    if any(p.kind == p.VAR_KEYWORD for p in params.values()):
        return kwargs
    bad_required = [k for k in required if k not in params]
    assert not bad_required, f"{fn} does not accept {bad_required}. Run help({fn.__name__}) and tell me what it shows."
    dropped = [k for k in kwargs if k not in params]
    if dropped:
        print("note: this library version ignores", dropped)
    return {k: v for k, v in kwargs.items() if k in params}


tsp_kwargs = dict(
    id_columns=[ID_COL], timestamp_column=TIME_COL, target_columns=["y"],
    control_columns=CONTROL_COLS, conditional_columns=CONDITIONAL_COLS,
    context_length=CONTEXT_LEN, prediction_length=PRED_LEN, scaling=False, encode_categorical=False,
)
REQUIRED = ["id_columns", "timestamp_column", "target_columns", "control_columns", "conditional_columns",
            "context_length", "prediction_length"]
tsp_kwargs = filter_kwargs(TimeSeriesPreprocessor, tsp_kwargs, required=REQUIRED)

ts = np.sort(df[TIME_COL].unique())
n_ids, n_total = df[ID_COL].nunique(), len(ts)
i_train_end = int(np.searchsorted(ts, np.datetime64(TRAIN_END, "ns")))
i_valid_end = int(np.searchsorted(ts, np.datetime64(VALID_END, "ns")))
W = CONTEXT_LEN + PRED_LEN


def make_split(offset):
    return {"train": [0, i_train_end],
            "valid": [i_train_end - offset, i_valid_end],
            "test": [i_valid_end - offset, n_total]}


expect = {
    "train": i_train_end - W + 1,
    "valid": (i_valid_end - i_train_end) - PRED_LEN + 1,
    "test": (n_total - i_valid_end) - PRED_LEN + 1,
}
chosen = None
for offset in (CONTEXT_LEN, 0):
    tsp = TimeSeriesPreprocessor(**tsp_kwargs)
    dset_train, dset_valid, dset_test = get_datasets(tsp, df, split_config=make_split(offset))
    got = {"train": len(dset_train) / n_ids, "valid": len(dset_valid) / n_ids, "test": len(dset_test) / n_ids}
    print(f"offset={offset}: windows per county {got}  expected {expect}")
    if all(abs(got[k] - expect[k]) <= 1 for k in expect):
        chosen = offset
        break
assert chosen is not None, "window counts match neither layout; send me the two lines printed above."
print("using split offset", chosen)

TARGET_IDX = list(tsp.prediction_channel_indices)[0]
print("input channels:", tsp.num_input_channels, "| target idx:", tsp.prediction_channel_indices,
      "| exogenous idx:", tsp.exogenous_channel_indices)
assert len(tsp.exogenous_channel_indices) > 0, "no exogenous channels were registered; weather/grid would be ignored"
item = dset_test[0]
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in item.items()})

## 7. Baseline numbers and a speed probe

In [ ]:
import time
from torch.utils.data import DataLoader, Subset


def collect(dataset, tidx):
    # Streams through the dataset keeping only what evaluation needs (full contexts would not fit in RAM).
    n = len(dataset)
    last = np.empty(n, dtype=np.float32)
    fut = np.empty((n, PRED_LEN), dtype=np.float32)
    for i in range(n):
        it = dataset[i]
        last[i] = float(it["past_values"][-1, tidx])
        fut[i] = it["future_values"][:, tidx].numpy()
    return last, fut


test_eval = dset_test if TEST_STRIDE == 1 else Subset(dset_test, range(0, len(dset_test), TEST_STRIDE))
t0 = time.time()
last_s, truth_s = collect(test_eval, TARGET_IDX)
print(f"collected {len(test_eval):,} test windows in {time.time() - t0:.0f}s")


def simple_collate(batch):
    return {k: torch.stack([torch.as_tensor(b[k]) for b in batch]) for k in ("past_values", "future_values")}


def probe_speed(model, dataset, steps=15):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=simple_collate)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    model.train()
    t = time.time()
    for i, batch in enumerate(loader):
        loss = model(past_values=batch["past_values"], future_values=batch["future_values"]).loss
        loss.backward(); opt.step(); opt.zero_grad()
        if i + 1 == steps:
            break
    per = (time.time() - t) / steps
    print(f"{per:.2f}s per step -> {MAX_STEPS} steps is about {per * MAX_STEPS / 60:.0f} minutes")

## 8. Load the pretrained TTM

In [ ]:
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
from tsfm_public import count_parameters
from transformers import set_seed

set_seed(SEED)
MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
model_kwargs = dict(
    num_input_channels=tsp.num_input_channels,
    prediction_channel_indices=tsp.prediction_channel_indices,
    exogenous_channel_indices=tsp.exogenous_channel_indices,
    decoder_mode="mix_channel",
)
try:
    model = get_model(MODEL_PATH, context_length=CONTEXT_LEN, prediction_length=PRED_LEN, **model_kwargs)
except TypeError as e:
    print("get_model rejected the arguments, loading directly instead:", e)
    model = TinyTimeMixerForPrediction.from_pretrained(MODEL_PATH, **model_kwargs)

if FREEZE_BACKBONE:
    for p in model.backbone.parameters():
        p.requires_grad = False
print("parameters:", count_parameters(model))
probe_speed(model, dset_train)

Check the speed line above before training. If it predicts more time than you have, lower `MAX_STEPS`
in the configuration cell and re-run from section 8. Re-running this cell reloads fresh pretrained weights.

## 9. Fine-tune

In [ ]:
import math
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from tsfm_public import TrackingCallback

EVAL_STEPS = max(50, MAX_STEPS // 10)
eval_kw = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters \
    else "evaluation_strategy"
args = TrainingArguments(
    output_dir=str(OUT_DIR / "trainer_output"), overwrite_output_dir=True,
    learning_rate=LEARNING_RATE, max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=256,
    dataloader_num_workers=0, report_to="none",
    save_strategy="steps", save_steps=EVAL_STEPS, logging_steps=EVAL_STEPS, save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    seed=SEED, **{eval_kw: "steps", "eval_steps": EVAL_STEPS},
)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
scheduler = OneCycleLR(optimizer, max_lr=LEARNING_RATE, total_steps=MAX_STEPS)
trainer = Trainer(
    model=model, args=args, train_dataset=dset_train, eval_dataset=dset_valid,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4), TrackingCallback()],
    optimizers=(optimizer, scheduler),
)
trainer.train()

In [ ]:
import json
model.save_pretrained(OUT_DIR / "model")
(OUT_DIR / "scaling.json").write_text(json.dumps({
    "y_mean": Y_MEAN, "y_std": Y_STD, "columns": SCALE_COLS,
    "stats": stats.to_dict(), "context_len": CONTEXT_LEN, "pred_len": PRED_LEN,
    "target_transform": "log1p(pct_out)", "model_path": MODEL_PATH, "synthetic": USE_SYNTHETIC,
}, indent=2))
print("saved to", OUT_DIR)

## 10. Evaluate on the held-out Uri month

Event = at least `EVENT_PCT` percent of a county's customers out at some point in the next 6, 24 or 96
hours. `auc` measures how well the forecast separates event windows from quiet ones (0.5 is chance).
`hit_rate` flags exactly as many top-scoring windows as there are real events and reports the share that were
truly events (1.0 is perfect; compare it with `event_rate`, which is what random flagging would score).

In [ ]:
out = trainer.predict(test_eval)
pred = out.predictions[0] if isinstance(out.predictions, (tuple, list)) else out.predictions
pred_s = np.asarray(pred)[..., 0] if np.asarray(pred).ndim == 3 else np.asarray(pred)
assert pred_s.shape == truth_s.shape, f"shape mismatch: predictions {pred_s.shape} vs truth {truth_s.shape}"

report = eval_report(pred_s, last_s, truth_s, Y_MEAN, Y_STD, EVENT_PCT)
print(report.round(3).to_string(index=False))
np.savez_compressed(OUT_DIR / "test_predictions.npz", pred_scaled=pred_s, truth_scaled=truth_s, last_scaled=last_s)

**How to read it:** the TTM only counts as working if it beats `persistence` at the 6h and 24h horizons
on `auc` and `hit_rate`. If persistence wins, the model is not adding value yet. Try
`FREEZE_BACKBONE = False`, more steps, or compare against a gradient-boosted model before changing the story.

**Not done yet (next steps):** per-county rankings and lead time on a rolling Uri replay need the county id
and timestamp of each window. Those come after the real data table exists.